# FRR flood reduction on Colab

This notebook measures FRR's IS-IS distributed flooding reduction ([FRR pull request 24128](https://github.com/FRRouting/frr/pull/24128), draft-ietf-lsr-distoptflood) on a small emulated network, with flood reduction off and then on. It uses [topology-builders](https://github.com/ahb-sjsu/topology-builders) and needs no Docker: every router is a network namespace of the Colab machine running real FRR daemons, and every link is a veth pair.

- **Runtime:** a CPU runtime is enough; the GPU of a GPU runtime is not used. More CPUs allow larger labs.
- **Time:** setup builds FRR from source, which takes several minutes. One small lab, off and on, takes a few more.
- **What carries over:** LSP counts and convergence outcomes. CPU time on a shared cloud machine is noisy, so compare off against on within one run rather than with numbers from other machines.

## 1. Check the runtime

The labs need a user namespace in which the notebook is allowed to create network devices. The last line says whether this runtime allows it.

In [ ]:
!echo "CPUs: $(nproc)"; free -g | head -2
!unshare -Urn sh -c 'ip link add tbcheck0 type veth peer name tbcheck1' && echo 'user namespaces: OK, the labs can run here' || echo 'this runtime does not allow user namespaces: the labs cannot run here'

## 2. Set up

Clones topology-builders, builds FRR from the pull request's commit with only the daemons the labs use (mgmtd, zebra, staticd and isisd), and installs both. Run it once per runtime.

In [ ]:
!git clone -q --depth 1 https://github.com/ahb-sjsu/topology-builders /content/topology-builders 2>/dev/null || git -C /content/topology-builders pull -q
!bash /content/topology-builders/tools/colab/setup.sh

## 3. Choose a topology

`FAMILY` and `ARGS` are the arguments of `python -m topology_builders build`, for example:

| Family | Arguments | Routers |
|---|---|---|
| `random-regular` | `--routers 16 --degree 4 --seed 1` | 16 |
| `fat-tree` | `--k 4` | 20 |
| `leaf-spine` | `--leaves 12 --spines 4` | 16 |
| `flattened-butterfly` | `--radices 4,4` | 16 |

Start small. Each router runs four FRR daemons, and a lab that is too large for the machine does not converge.

In [ ]:
FAMILY = "random-regular"
ARGS = "--routers 16 --degree 4 --seed 1"
NAME = "lab1"   # a new name for each topology
LINKS = 2       # link changes (down, then up) per run
REPEAT = 1      # runs of each lab; the order of off and on alternates

!python -m topology_builders build {FAMILY} {ARGS} --out {NAME}.json
!python -m topology_builders clab {NAME}.json --out labs/{NAME} --name {NAME} --timing-logs

## 4. Measure

For each lab, flood reduction off and then on: start the routers, wait for IS-IS to converge, take each link down and up, and record the LSPs sent, isisd CPU per event handler and the time until every router has the change. `unshare -Urnm` gives the run its own user, network and mount namespaces, so nothing is added to the Colab machine's own network.

In [ ]:
!unshare -Urnm python -m topology_builders measure labs/{NAME}/off labs/{NAME}/on --backend netns --links {LINKS} --repeat {REPEAT} --settle 20 --max-wait 180 --timing --profile --out measurements.jsonl

## 5. Report

One row per topology: LSPs sent and isisd CPU per link change with flood reduction off and on, and the saving. The repository's `results/` folder has the same tables from 64-router runs.

In [ ]:
!python -m topology_builders report measurements.jsonl

In [ ]:
# Download the raw measurements (JSON lines, one record per run).
try:
    from google.colab import files
    files.download("measurements.jsonl")
except ImportError:
    import os
    print("measurements.jsonl is in", os.getcwd())